# SKEX on Kaggle

Upload this notebook to Kaggle with **Internet on** and **one T4**. Do not switch on a second T4. Dual T4 spends the weekly quota twice and this project does not use it.

The notebook clones `https://github.com/umardrazbhatti-work/skex` and runs the CPU smoke gate. It does **not** download a 7B model, install Unsloth, or start plan `01` / QLoRA.

Optional: add the `Dataset/` folder you uploaded as a Kaggle dataset, then run the later cells. Those cells copy Domain-A JSONL into the clone, or build it from SciRIFF parquet if the processed files are not in the upload.

In [ ]:
import shutil
import subprocess
from pathlib import Path

if shutil.which("nvidia-smi"):
    listing = subprocess.check_output(["nvidia-smi", "-L"], text=True)
    print(listing)
    gpus = [line for line in listing.splitlines() if line.strip()]
    if len(gpus) != 1:
        raise SystemExit(f"Stop. Expected one T4, found {len(gpus)}. Turn the extra GPU off before any training run.")
    if "T4" not in listing:
        print("WARNING: the visible GPU is not named T4. Do not start a larger run on this session.")
else:
    print("No NVIDIA GPU visible. The smoke cells below do not need one.")

In [ ]:
import subprocess
import sys
from pathlib import Path

REMOTE = "https://github.com/umardrazbhatti-work/skex.git"

def run(cmd, cwd):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd))

if Path("/kaggle/working").is_dir():
    work = Path("/kaggle/working")
    repo = work / "skex"
    if not (repo / ".git").exists():
        run(["git", "clone", REMOTE], work)
    else:
        run(["git", "pull", "--ff-only"], repo)
elif (Path.cwd() / "src" / "skex").is_dir():
    repo = Path.cwd()
    print(f"Already inside {repo}. Not cloning.")
else:
    raise SystemExit("Run this notebook on Kaggle, or from the skex repo root.")

run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], repo)
print("repo", repo)

In [ ]:
import shutil
import sys
import subprocess
from pathlib import Path

repo = Path("/kaggle/working/skex") if Path("/kaggle/working/skex").is_dir() else Path.cwd()
inputs = Path("/kaggle/input")
dest = repo / "data" / "processed" / "domain_a"
dest.mkdir(parents=True, exist_ok=True)

copied = False
if inputs.is_dir():
    hits = sorted(inputs.glob("**/domain_a/train.jsonl"))
    print("processed train files:", hits)
    if hits:
        for name in ("train.jsonl", "dev.jsonl", "test.jsonl"):
            src = hits[0].parent / name
            if src.exists():
                shutil.copy2(src, dest / name)
                print("copied", src)
                copied = True
    if not copied:
        parquet = sorted(inputs.glob("**/4096/*.parquet"))
        print("SciRIFF parquet files:", parquet)
        if parquet:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "pyarrow"],
                cwd=str(repo),
            )
            subprocess.check_call(
                [sys.executable, "-m", "skex.data.convert_sciriff", "--src", str(parquet[0].parent), "--dest", str(dest)],
                cwd=str(repo),
            )
            copied = True
else:
    print("No /kaggle/input. Attach the uploaded Dataset folder if you want Domain-A JSONL in this session.")

if not copied:
    print("No Domain-A JSONL yet. Smoke still runs on the built-in example record.")

In [ ]:
import subprocess
import sys
from pathlib import Path

repo = Path("/kaggle/working/skex") if Path("/kaggle/working/skex").is_dir() else Path.cwd()
subprocess.check_call([sys.executable, "-m", "pytest", "-q"], cwd=str(repo))
for _ in range(2):
    subprocess.check_call(
        [sys.executable, "-m", "skex.experiments.runner", "--plan", "experiments/plans/00_smoke.yaml"],
        cwd=str(repo),
    )
subprocess.check_call([sys.executable, "-m", "skex.experiments.status"], cwd=str(repo))
print("Stop here. Do not run experiments/plans/01_tax_zeroshot.yaml or install Unsloth in this notebook until you choose to spend T4 quota.")